# 🔬 Quantitative Measure Governance Lab (Groq Edition)

This notebook implements the **Quantitative 'Research Loop'** to validate metric governance contracts against the literature, using the **Groq** LLM API.

## Research Loop Objectives
1. **Hypothesis/Metric Selection**: Test candidates like 'CRB/CRLB', 'RMSE', 'Sensing Bandwidth'.
2. **Contract Filter**: Check compliance with metric governance (e.g., is it $\Delta r_{min}$ or $\sigma_r$?)
3. **Targeted Search**: Locate evidence and log specific headings/locators.
4. **Ambiguity Assessment**: Classify as 'AMBIGUOUS' or 'DEFENSIBLE'.

## User Context
> "A.1 'ölçüm düzlemi' kontratını kilitliyoruz; ama literatürde aynı hedefi farklı metrik isimleriyle yapmaya çalışan çalışmalar da var. Bu yüzden kontrollü bir 'araştırma döngüsü' kuralım."

**Update:**
1. Full-text analysis (no truncation).
2. Scans `data/proc_markdowns` (Source Text) recursively.
3. References `data/ext_res_v4` (Existing Extractions).

In [32]:
# @title 1. Install Dependencies
!pip install -q groq

In [33]:
# @title 2. Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Define Root Path (Adjust if needed)
BASE_DIR = "/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST"
os.chdir(BASE_DIR)
print(f"📂 Working Directory set to: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Working Directory set to: /content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST


In [34]:
# @title 3. Load Groq API Key
from google.colab import userdata
import os

try:
    # Ensure you have added 'GROQ_API_KEY' to your Colab Secrets (Key icon on the left)
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("🔑 Groq API Key loaded successfully.")
except Exception as e:
    print(f"❌ Error loading API Key: {e}\nPlease ensure 'GROQ_API_KEY' is set in Colab Secrets.")

🔑 Groq API Key loaded successfully.


In [35]:
# @title 4. Section II-A Evidence Agent (Planes + Observation Models)
from groq import Groq
import json
import re

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

RESPONSE_SCHEMA = {
    "evidence": [
        {
            "metric": "OSNR | ESNR | SNR | COHERENT_MODEL | IMDD_MODEL",
            "plane": "OPTICAL_PLANE | ELECTRICAL_PLANE | MIXED_PLANE | AMBIGUOUS | N/A",
            "quote": "<=25 words excerpt",
            "heading_path": "HeadingPath or unknown_heading",
            "line_start": 0,
            "line_end": 0,
            "strength": "strong | weak",
            "rationale": "short evidence-based rationale"
        }
    ],
    "notes": "short notes or empty"
}


def _parse_json_response(text):
    if not text:
        raise ValueError("empty response")
    try:
        return json.loads(text)
    except Exception:
        fenced = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, flags=re.DOTALL)
        if fenced:
            return json.loads(fenced.group(1))
        block = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if block:
            return json.loads(block.group(0))
        raise


def analyze_section2A_evidence(snippets, paper_id="Unknown"):
    """
    Given snippet candidates (from full-document scan), extract valid evidence for:
    - Measurement planes (OSNR vs electrical SNR)
    - Observation models (coherent vs IM/DD)
    """
    system_prompt = """
You are an Evidence-Locked Technical Writer and Consistency Auditor for Section II-A.
Your task is to extract evidence for (i) measurement-plane separation and (ii) observation models.

Rules:
- Use ONLY the provided snippets as evidence.
- OSNR evidence must be explicitly optical (OSNR / optical SNR / optical-domain cues).
- ESNR evidence must be explicitly electrical/post-detection/baseband (electrical SNR, post-detection, receiver output, IM/DD SNR).
- Generic 'SNR' without optical/electrical cues is AMBIGUOUS.
- Coherent model evidence must mention coherent detection/receiver, complex baseband, or LO/IQ terminology.
- IM/DD model evidence must mention IM/DD, intensity modulation, direct detection, photodiode/photodetector, responsivity, or nonnegativity.
- Quote must be <=25 words and taken verbatim from the snippet.
- Set plane = N/A for model evidence (COHERENT_MODEL / IMDD_MODEL).
Return STRICT JSON using the given schema, no extra keys.
"""

    user_prompt = f"""
Paper ID: {paper_id}

Candidate snippets (each has heading_path + line numbers + strength + candidate label):
{json.dumps(snippets, ensure_ascii=False, indent=2)}

Return JSON exactly in this schema (no extra keys):
{json.dumps(RESPONSE_SCHEMA, ensure_ascii=False, indent=2)}
"""

    try:
        completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            model="llama-3.3-70b-versatile",
            temperature=0.2
        )
        return _parse_json_response(completion.choices[0].message.content)
    except Exception as e:
        return {"evidence": [], "notes": f"Error or non-JSON response: {e}"}


In [36]:
# @title 5. Data Loader + Section II-A Snippet Extractor (Recursive)
import glob
import os
import re

EVIDENCE_PATTERNS = {
    "OSNR": {
        "strong": [
            r"\bOSNR\b",
            r"\bO-?SNR\b",
            r"optical signal[- ]?to[- ]?noise ratio",
            r"optical signal to noise ratio",
            r"optical SNR",
            r"optical-domain SNR",
        ],
        "weak": [
            r"optical.*SNR",
            r"SNR.*optical",
        ],
    },
    "ESNR": {
        "strong": [
            r"electrical SNR",
            r"electrical[- ]domain SNR",
            r"post[- ]detection SNR",
            r"after detection SNR",
            r"received electrical SNR",
            r"IM/DD SNR",
            r"baseband SNR",
            r"SNR at the receiver output",
        ],
        "weak": [
            r"photodetector.*SNR",
            r"PD output.*SNR",
            r"after photodiode.*SNR",
            r"receiver output.*SNR",
        ],
    },
    "SNR": {
        "strong": [],
        "weak": [r"\bSNR\b", r"signal[- ]to[- ]noise ratio"],
    },
    "COHERENT_MODEL": {
        "strong": [
            r"coherent detection",
            r"coherent receiver",
            r"optical coherent",
            r"heterodyne",
            r"homodyne",
            r"local oscillator",
            r"\bI/Q\b",
            r"IQ receiver",
            r"complex baseband",
            r"intradyne",
        ],
        "weak": [
            r"coherent",
            r"DSP.*coherent",
        ],
    },
    "IMDD_MODEL": {
        "strong": [
            r"IM/DD",
            r"intensity modulation direct detection",
            r"intensity modulation",
            r"direct detection",
            r"photodiode",
            r"photodetector",
            r"responsivity",
            r"nonnegative",
            r"square[- ]law",
        ],
        "weak": [
            r"IMDD",
            r"DD receiver",
            r"intensity.*detection",
        ],
    },
}

HEADING_RE = re.compile(r"^#+\s+")
REF_RE = re.compile(r"references|bibliography|\bref\.\b", re.IGNORECASE)


def index_headings(lines):
    path = []
    heading_at = [""] * (len(lines) + 1)
    for i, line in enumerate(lines, 1):
        if HEADING_RE.match(line):
            heading = line.strip()
            level = len(heading.split()[0])
            while len(path) >= level:
                path.pop()
            path.append(heading)
        heading_at[i] = " > ".join(path)
    return heading_at


def is_reference_heading(heading_path):
    return bool(REF_RE.search(heading_path or ""))


def extract_section2A_snippets(lines, heading_at, window=2, max_snippets=30):
    """Scan full paper and return candidate snippets for Section II-A evidence."""
    snippets = []
    used_keys = set()
    for metric, groups in EVIDENCE_PATTERNS.items():
        for strength in ["strong", "weak"]:
            for pat in groups.get(strength, []):
                for i, line in enumerate(lines, 1):
                    if re.search(pat, line, re.IGNORECASE):
                        if is_reference_heading(heading_at[i]):
                            continue
                        start = max(1, i - window)
                        end = min(len(lines), i + window)
                        key = (start, end, heading_at[i], metric, strength)
                        if key in used_keys:
                            continue
                        excerpt = " ".join(l.strip() for l in lines[start-1:end] if l.strip())
                        snippets.append({
                            "metric": metric,
                            "strength": strength,
                            "heading_path": heading_at[i] or "unknown_heading",
                            "line_start": start,
                            "line_end": end,
                            "excerpt": excerpt
                        })
                        used_keys.add(key)
                        if len(snippets) >= max_snippets:
                            return snippets
    return snippets


def load_papers(base_dir="data/proc_markdowns", target_ids=None, limit=None):
    patterns = [
        os.path.join(base_dir, "**", "*.md"),
        os.path.join(base_dir, "*.md")
    ]
    all_files = []
    for p in patterns:
        all_files.extend(glob.glob(p, recursive=True))
    all_files = list(set(all_files))

    selected_files = []
    if target_ids:
        print(f"Applying filter for {len(target_ids)} Target IDs...")
        for f_path in all_files:
            p_id = os.path.basename(f_path).replace('.md', '')
            if p_id in target_ids:
                selected_files.append(f_path)
    else:
        selected_files = all_files

    if limit:
        selected_files = selected_files[:limit]

    print(f"Found {len(all_files)} total markdown files. Loading content for {len(selected_files)} selected files.")

    data = []
    for f_path in selected_files:
        p_id = os.path.basename(f_path).replace('.md', '')
        with open(f_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.read().splitlines()
        heading_at = index_headings(lines)
        data.append((p_id, lines, heading_at))
    return data


In [37]:
# @title 6. Execute Section II-A Evidence Scan
import csv

# ==========================================
# USER CONFIGURATION AREA
# ==========================================

TARGET_PAPERS = []  # Example: ['O_ISAC_023', 'O_ISAC_061']
LIMIT_COUNT = None  # Full run (all papers)

OUTPUT_CSV = "analysis/II_ev_v2/section2A_evidence.csv"

# ==========================================

paper_batch = load_papers(base_dir="data/proc_markdowns", target_ids=TARGET_PAPERS, limit=LIMIT_COUNT)

print("\nStarting Section II-A evidence scan...\n")

rows = []
for p_id, lines, heading_at in paper_batch:
    print(f"=== Scanning: {p_id} ({len(lines)} lines) ===")
    snippets = extract_section2A_snippets(lines, heading_at, window=2, max_snippets=30)

    if not snippets:
        continue

    result = analyze_section2A_evidence(snippets, p_id)
    for ev in result.get("evidence", []):
        line_start = ev.get("line_start", "")
        heading_path = ev.get("heading_path") or (heading_at[line_start] if isinstance(line_start, int) and line_start < len(heading_at) else "unknown_heading")
        rows.append({
            "paper_id": p_id,
            "metric": ev.get("metric", ""),
            "plane": ev.get("plane", ""),
            "strength": ev.get("strength", ""),
            "quote": ev.get("quote", ""),
            "heading_path": heading_path,
            "line_start": ev.get("line_start", ""),
            "line_end": ev.get("line_end", ""),
            "rationale": ev.get("rationale", "")
        })

    print("\n" + "="*40 + "\n")

if rows:
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved: {OUTPUT_CSV}")
else:
    print("No evidence rows produced.")


Found 313 total markdown files. Loading content for 313 selected files.

Starting Section II-A evidence scan...

=== Scanning: O_ISAC_276 (199 lines) ===


=== Scanning: O_ISAC_143 (110 lines) ===
=== Scanning: O_ISAC_032 (318 lines) ===
=== Scanning: O_ISAC_033 (79 lines) ===


=== Scanning: O_ISAC_166 (395 lines) ===


=== Scanning: O_ISAC_030 (416 lines) ===


=== Scanning: O_ISAC_062 (1105 lines) ===


=== Scanning: O_ISAC_039 (801 lines) ===


=== Scanning: O_ISAC_033 (89 lines) ===


=== Scanning: O_ISAC_200 (517 lines) ===


=== Scanning: O_ISAC_003 (202 lines) ===
=== Scanning: O_ISAC_021 (234 lines) ===


=== Scanning: O_ISAC_111 (115 lines) ===


=== Scanning: O_ISAC_007 (155 lines) ===
=== Scanning: O_ISAC_130 (366 lines) ===


=== Scanning: O_ISAC_049 (795 lines) ===


=== Scanning: O_ISAC_121 (45 lines) ===


=== Scanning: O_ISAC_218 (386 lines) ===


=== Scanning: O_ISAC_019 (150 lines) ===


=== Scanning: O_ISAC_123 (348 lines) ===
=== Scanning: O_ISAC_097 (296 lines) ==